[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/warlog7997/rag-playbook/blob/master/learn/01-what-is-rag/notebook.ipynb)

# Chapter 1 — What is RAG?

## The Problem RAG Solves

You have an LLM that knows everything about the world up to its training cutoff.
But it knows **nothing** about *your* product, *your* docs, *your* codebase.

Ask Claude *"How do I reset my password in NovaDash?"* — it will either say it doesn't know, or worse, **make something up**.

**RAG fixes this by giving the LLM an open-book exam instead of a closed one.**

Instead of relying on training data, you:
1. **Index** your documents into a vector database
2. **Retrieve** the most relevant chunks at query time
3. **Generate** an answer grounded in those chunks

---

## The Knowledge Base

All examples in this notebook use the **NovaDash** documentation — a fictional SaaS product with ~10 markdown files covering billing, authentication, user management, projects, integrations, and more. Simple enough to understand quickly, real enough to make RAG interesting.

---

## What You'll Learn
- What embeddings are and why they work
- How ChromaDB stores and queries vectors
- The 3-step Retrieve → Generate pipeline
- What happens when you ask about something NOT in the docs

## Setup

Run the cell below to install dependencies and configure your API key.

**Requirements:**
- Python 3.9+
- An Anthropic API key (get one at [console.anthropic.com](https://console.anthropic.com))
- The NovaDash knowledge base must be ingested first (run the ingest cell below)

**First time only:** Run `python learn/ingest.py` from the repo root, or run the ingest cell in this notebook.

In [ ]:
# Install required packages
!pip install anthropic==0.49.0 chromadb sentence-transformers rank-bm25 keyring -q

In [ ]:
import os
os.environ['PYTHONUTF8'] = '1'

# Set your Anthropic API key
# Option 1: keyring (recommended — stores securely in OS keychain)
import keyring
# keyring.set_password("Anthropic", "API_KEY", "your-key-here")  # run once to store

# Option 2: environment variable
# os.environ['ANTHROPIC_API_KEY'] = 'your-key-here'

import sys
from pathlib import Path

# Add learn/ to path so we can import utils.py
REPO_ROOT = Path.cwd().parent.parent  # adjust if running from a different directory
LEARN_DIR = REPO_ROOT / 'learn'
sys.path.insert(0, str(LEARN_DIR))

print(f"Repo root: {REPO_ROOT}")
print(f"Learn dir: {LEARN_DIR}")
print("Setup complete!")

## The 3-Step RAG Pipeline

Before writing any code, let's understand the architecture:

```
Your Question
     │
     ▼
[1. INDEX]  ──── Your docs are embedded into vectors and stored in ChromaDB
     │            (done once, at ingest time)
     │
     ▼
[2. RETRIEVE] ── Your question is embedded, similar chunks are fetched
     │
     ▼
[3. GENERATE] ── Claude reads the chunks + your question → answers it
     │
     ▼
  Answer (grounded in your actual docs)
```

**Key insight:** RAG is essentially *context window stuffing with a search filter*. Instead of putting your entire knowledge base into the prompt (expensive, slow), you retrieve only the top 3-5 most relevant chunks.

## Step 1: Ingest the Docs

Before we can retrieve anything, we need to:
1. Load the NovaDash markdown files
2. Split them into sections (chunks)
3. Generate an embedding vector for each chunk
4. Store everything in ChromaDB + a BM25 index

This only needs to run **once** — the indexes are saved to disk.

In [ ]:
# Run ingestion — only needed once.
# This reads all .md files in knowledge-base/, chunks them by ## headers,
# embeds each chunk with all-MiniLM-L6-v2, and saves to:
#   data/chromadb/   (vector store)
#   data/bm25_index.pkl  (BM25 index for keyword search)

import subprocess
result = subprocess.run(
    [sys.executable, str(LEARN_DIR / 'ingest.py')],
    capture_output=True, text=True, cwd=str(REPO_ROOT)
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

## What Are Embeddings?

An **embedding** is a dense numerical vector that represents the *meaning* of text. Similar meanings produce similar vectors — small geometric distance in vector space.

We use `all-MiniLM-L6-v2` which produces **384-dimensional vectors**. Every sentence becomes a point in 384-dimensional space. Semantically similar sentences cluster together.

Let's see this in action:

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

# Load the same model used during ingest
model = SentenceTransformer("all-MiniLM-L6-v2")

# Embed several sentences
sentences = [
    "How do I reset my password?",
    "I forgot my account credentials",
    "Account recovery via email verification token flow",  # how the DOCS phrase it
    "How do I create a project?",  # completely different topic
]

embeddings = model.encode(sentences)
print(f"Embedding shape: {embeddings.shape}  (4 sentences × 384 dimensions)\n")

# Compute cosine similarity between all pairs
def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

print("Cosine similarities (1.0 = identical meaning, 0.0 = unrelated):\n")
for i, s1 in enumerate(sentences):
    for j, s2 in enumerate(sentences):
        if i < j:
            sim = cosine_similarity(embeddings[i], embeddings[j])
            print(f"  {sim:.3f}  '{s1[:40]}' vs '{s2[:40]}'")

Notice that "reset my password" and "Account recovery via email verification token flow" have a **higher similarity** than you might expect — even though they use completely different words. This is what makes semantic search powerful.

## Step 2: Retrieve Relevant Chunks

Now let's use the utils from the repo to search ChromaDB:

In [ ]:
from utils import vector_search, format_results

# The question we want to answer
QUESTION = "How do I reset my password in NovaDash?"

print(f"Question: {QUESTION}\n")
print("Step 1: Searching for relevant docs via vector similarity...\n")

# Search ChromaDB: embed the question, find top-3 nearest chunks
hits = vector_search(QUESTION, top_k=3)

# Display the results
format_results(hits, title="Retrieved Chunks")

## Step 3: Build Context + Generate Answer

Now we take the retrieved chunks, format them as context, and ask Claude to answer using **only** that context. This is the grounding step — the model cannot hallucinate beyond what we provide.

In [ ]:
from utils import ask_claude

# Format the retrieved chunks into a readable context block
context = "\n\n---\n\n".join([
    f"Source: {h['source']} | Section: {h['section']}\n{h['text']}"
    for h in hits
])

# System prompt: tell Claude to ONLY use the provided context
system_prompt = """You are a helpful support assistant for NovaDash.
Answer the user's question using ONLY the provided documentation context.
If the answer isn't in the context, say "I don't have enough information to answer that."
Be concise and direct."""

user_prompt = f"""Documentation context:
{context}

Question: {QUESTION}"""

print("Step 2: Asking Claude to answer using retrieved context...\n")
answer = ask_claude(system=system_prompt, user=user_prompt, model="claude-haiku-4-5-20251001")

print("=" * 60)
print("  Answer")
print("=" * 60)
print(answer)
print()

The answer is **grounded in the actual docs** — Claude can only say what the retrieved chunks tell it.

## Testing with an Unknown Question

What happens when we ask about something NOT in the NovaDash documentation? The model should correctly say it doesn't know — rather than inventing an answer.

In [ ]:
# Test: ask about something that's not in the docs at all
unknown_question = "What is NovaDash's office address?"

hits2 = vector_search(unknown_question, top_k=3)
context2 = "\n\n---\n\n".join([
    f"Source: {h['source']} | Section: {h['section']}\n{h['text']}"
    for h in hits2
])

answer2 = ask_claude(
    system=system_prompt,
    user=f"Documentation context:\n{context2}\n\nQuestion: {unknown_question}"
)

print("=" * 60)
print("  Testing: Unknown Question")
print("=" * 60)
print(f"Question: {unknown_question}")
print()
print("Top retrieved chunks (even though they don't contain the answer):")
for i, hit in enumerate(hits2[:3], 1):
    print(f"  [{i}] {hit['source']} > {hit['section']}  (score: {hit['score']:.4f})")

print()
print(f"Answer:\n{answer2}")

The system correctly declines to answer — because the system prompt enforces grounding.

## Full RAG Pipeline in One Function

Let's wrap everything into a clean, reusable function that shows the full pipeline:

In [ ]:
from utils import vector_search, ask_claude

def simple_rag(question: str, top_k: int = 3) -> dict:
    """
    The simplest possible RAG pipeline:
      1. Embed the question
      2. Retrieve top_k most similar chunks from ChromaDB
      3. Ask Claude to answer using ONLY those chunks
    """
    # Step 1 + 2: Retrieve
    hits = vector_search(question, top_k=top_k)

    # Step 3: Build context and generate
    context = "\n\n---\n\n".join([
        f"Source: {h['source']} | Section: {h['section']}\n{h['text']}"
        for h in hits
    ])

    answer = ask_claude(
        system="""You are a helpful NovaDash support assistant.
Answer using ONLY the documentation context provided.
If the answer isn't in the context, say 'I don't have enough information.'""",
        user=f"Documentation context:\n{context}\n\nQuestion: {question}",
        model="claude-haiku-4-5-20251001",
    )

    return {"question": question, "hits": hits, "answer": answer}


# Try a few questions
test_questions = [
    "How do I add a new team member?",
    "What are the limits on the free plan?",
    "How does authentication work?",
]

for q in test_questions:
    result = simple_rag(q)
    print(f"Q: {q}")
    print(f"Sources: {[h['source'] + ' > ' + h['section'] for h in result['hits'][:2]]}")
    print(f"A: {result['answer'][:200].strip()}...\n")

## Key Insight: When Naive Vector Search Fails

Notice that in our very first test, "reset my password" worked — but it bridged the gap because the intro section happened to mention both concepts.

This doesn't **always** work. Run the cell below to see an example where naive vector search fails:

In [ ]:
# The NovaDash docs use "subscription termination" — NOT "cancel"
# Watch what naive vector search returns for a user who says "cancel"

query = "How do I cancel my subscription?"
hits = vector_search(query, top_k=3)

print(f"Query: '{query}'")
print(f"Expected: billing.md > subscription termination section")
print()
print("What naive vector search returns:")
for i, hit in enumerate(hits, 1):
    is_correct = "billing" in hit['source'] and "terminat" in hit['section'].lower()
    marker = "CORRECT" if is_correct else "WRONG  "
    print(f"  [{i}] {marker} {hit['source']} > {hit['section']}  (score: {hit['score']:.4f})")

print()
print("The docs say 'subscription termination' — NOT 'cancel'.")
print("Vector search found semantically related chunks, but missed the exact answer.")
print("\n>> See Chapter 2 to fix this with hybrid search.")

## Summary

You've built the simplest possible RAG system:

1. **Embed** — Convert text to vectors using `all-MiniLM-L6-v2`
2. **Store** — ChromaDB stores vectors + text for fast nearest-neighbor search
3. **Retrieve** — Embed the question, find the closest chunk vectors
4. **Generate** — Claude reads the chunks and answers only from them

**What works:**
- Questions that use similar words to the docs
- Clear factual queries within the knowledge base
- "I don't know" responses for out-of-scope questions

**What breaks:**
- User says "cancel" but docs say "termination"
- User says "remove user" but docs say "member offboarding"
- Questions that span multiple sections

These are exactly the problems we fix in the next chapters.

---

**Next:** [Chapter 2 — My Bot Returns Wrong Answers](../02-wrong-answers/notebook.ipynb)

We'll see why naive vector search fails on semantic gaps and fix it with BM25 + Reciprocal Rank Fusion (hybrid search).